In [ ]:
# DATA HANDLING + MATH
import pandas as pd
import numpy as np

# PATH & TIME
from pathlib import Path
from datetime import datetime
import pytz
from google.colab import drive

# VISUALIZATION
import matplotlib.pyplot as plt
import seaborn as sns

# MACHINE LEARNING
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import adjusted_rand_score, accuracy_score, confusion_matrix, mean_squared_error, mean_absolute_error
from scipy.optimize import linear_sum_assignment

# PARALLEL RUN
from joblib import Parallel, delayed
import multiprocessing


In [ ]:
drive.mount('/content/drive')

folder_path = Path('/content/drive/MyDrive/folder_path')
out_folder  = Path('/content/drive/MyDrive/out_folder')
folder_path.mkdir(parents=True, exist_ok=True)
out_folder.mkdir(parents=True, exist_ok=True)

print(f'Input  folder: {folder_path}')
print(f'Output folder: {out_folder}')


In [ ]:
def _prep_df(df, cols2drop, hz=8):
  # Filter df to include only the first FORECAST_HORIZON_WEEKS of sales for each product
  df['weeks_since_first_sale'] = (df['invoice_date'] - df['first_sale']).dt.days // 7 + 1
  df = df[df['weeks_since_first_sale'] <= FORECAST_HORIZON_WEEKS].copy()
  df = df.drop(columns=['weeks_since_first_sale'])

  print(f'Filtered to {len(df):,} rows, representing the first {FORECAST_HORIZON_WEEKS} weeks of sales for each product.')

  df['item_quantity']   = df['item_quantity'].astype(int)

  # Pure label concatenations — no aggregation, no leakage
  df['full_sub_cat']     = df['Category'] + ' - ' + df['sub cat']
  df['full_sub_sub_cat'] = df['full_sub_cat'] + ' - ' + df['sub sub cat']

  # NEW BRAND FEATURE
  # Calculate the global first sale date for each brand
  brand_launches = df.groupby('brand_name')['first_sale'].min().rename('brand_first_launch').reset_index()
  df = df.merge(brand_launches, on='brand_name', how='left')
  # A product is part of a "new brand" if its launch date is the same as the brand's first ever launch date
  df['is_new_brand'] = (df['first_sale'] == df['brand_first_launch']).astype(int)
  df.drop(columns=['brand_first_launch'], inplace=True)

  # NUM_SIMILARS
  # 1. Get unique products with their launch dates and categories
  product_launches = df[['product_code', 'full_sub_sub_cat', 'first_sale']].drop_duplicates()

  # 2. Define a function to count predecessors
  def count_predecessors(row, all_launches):
      return all_launches[
          (all_launches['full_sub_sub_cat'] == row['full_sub_sub_cat']) &
          (all_launches['first_sale'] < row['first_sale'])
      ]['product_code'].nunique()

  # 3. Apply the logic
  unique_products = product_launches.copy()
  unique_products['num_similars_new'] = unique_products.apply(
      count_predecessors,
      axis=1,
      all_launches=unique_products
  )

  # 4. Merge back to main dataframe and replace the old column
  df = df.merge(unique_products[['product_code', 'num_similars_new']], on='product_code', how='left')
  df['num_similars'] = df['num_similars_new']
  df.drop(columns=['num_similars_new'], inplace=True)

  # SUBSTITUTABILITY
  def map_substitutability(n):
    if n <= 3:
        return 'subs_3'
    elif n <= 20:
        return 'subs_20'
    elif n <= 40:
        return 'subs_40'
    else:
        return 'subs_ultra'

  df['substitutability'] = df['num_similars'].apply(map_substitutability)

  # PRICE_INDEX
  inflation_rates = {
    2021: 0.402, 2022: 0.458, 2023: 0.407, 2024: 0.325, 2025: 0.509, 2026: 0.689
  }

  def get_cumulative_inflation_factor(target_year, target_month, base_year=2021):
      factor = 1.0
      for year in range(base_year, int(target_year)):
          factor *= (1 + inflation_rates.get(year, 0.0))
      annual_rate_target = inflation_rates.get(int(target_year), 0.0)
      monthly_rate_target = (1 + annual_rate_target)**(1/12) - 1
      factor *= (1 + monthly_rate_target) ** (int(target_month) - 1)
      return float(factor)

  mask = (df['unusual_sales'].astype(str).str.lower() != 'yes') & (df['item_quantity'] > 0)
  df_filtered = df[mask].copy()

  if len(df_filtered) > 0:
      df_filtered['year'] = df_filtered['invoice_date'].dt.year
      df_filtered['month'] = df_filtered['invoice_date'].dt.month
      df_filtered['cum_inflation_factor'] = [get_cumulative_inflation_factor(y, m) for y, m in zip(df_filtered['year'], df_filtered['month'])]
      df_filtered['adj_unit_gross'] = df_filtered['unit_gross'] / df_filtered['cum_inflation_factor']
      new_price_indices = df_filtered.groupby('product_code')['adj_unit_gross'].mean().rename('price_index')
      df = df.drop(columns=['price_index'], errors='ignore').merge(new_price_indices, on='product_code', how='left')
  else:
      print("Warning: Filtered dataset is empty. Check 'unusual_sales' and 'item_quantity' values.")

  df_2 = df.drop(columns=cols2drop, errors='ignore')
  display(df_2.info())
  display(df_2.head())

  return df_2

In [ ]:
def _calculate_daily_product_activity(df_local):
    """Daily quantity + holiday flag per (product, date)."""
    daily = df_local.groupby(['product_code', 'product_name', 'invoice_date']).agg(
        daily_quantity=('item_quantity', 'sum'),
        is_holiday=('holiday', 'max')
    ).reset_index()
    daily['is_zero_quantity_day']        = (daily['daily_quantity'] == 0).astype(int)
    daily['is_zero_quantity_working_day'] = (
        (daily['daily_quantity'] == 0) & (daily['is_holiday'] == 0)
    ).astype(int)
    daily['is_transaction_day'] = (daily['daily_quantity'] > 0).astype(int)
    return daily


def _calculate_product_sale_dates(daily, products_base_subset):
    """
    Product-level tenure, working days, zero-sales share, and launch month.
    """
    psd = daily.groupby(['product_code', 'product_name']).agg(
        min_date=('invoice_date', 'min'),
        max_date=('invoice_date', 'max'),
        total_holiday_days=('is_holiday', 'sum'),
        count_zero_quantity_days=('is_zero_quantity_day', 'sum'),
        zero_sales_working_days=('is_zero_quantity_working_day', 'sum')
    ).reset_index()

    psd['duration_days']   = (psd['max_date'] - psd['min_date']).dt.days + 1
    psd['total_working_days'] = np.maximum(
        psd['duration_days'] - psd['total_holiday_days'], 0
    )
    psd['share_zero_qty_working_days'] = (
        psd['zero_sales_working_days'] / psd['total_working_days']
    ).fillna(0)

    psd['duration_weeks'] = np.minimum(np.maximum(psd['duration_days'] / 7.0, 1.0), float(FORECAST_HORIZON_WEEKS))

    # Add launch month as categorical feature
    psd = psd.merge(products_base_subset[['product_code', 'first_sale']], on='product_code', how='left')
    psd['launch_month'] = psd['first_sale'].dt.month.astype(str)
    psd = psd.drop(columns=['first_sale'])

    return psd


def _aggregate_core_features(df_local):
    """Total quantity, split-local price_index and transaction count."""
    core = df_local.groupby('product_code').agg(
        total_sales_quantity=('item_quantity', 'sum'),
        price_index=('price_index', 'max')
    ).reset_index()
    txn = (
        df_local[df_local['item_quantity'] > 0]
        .groupby('product_code').size()
        .reset_index(name='num_sales_transactions')
    )
    return core.merge(txn, on='product_code', how='left')


def _calculate_discount_rate(df_local):
    """Mean discount rate, restricted to non-zero quantity rows."""
    return (
        df_local[df_local['item_quantity'] > 0]
        .groupby('product_code')['discount_rate']
        .mean()
        .reset_index(name='median_discount_rate')
    )


def _calculate_cv_quantity(df_local):
    """Coefficient of variation and median of weekly sales quantities."""
    weekly = (
        df_local.set_index('invoice_date')
        .groupby('product_code')['item_quantity']
        .resample('W').sum()
        .reset_index()
    )
    cv = weekly.groupby('product_code')['item_quantity'].agg(['mean', 'std', 'median']).reset_index()
    cv.rename(columns={'median': 'median_weekly_quantity'}, inplace=True)
    cv['cv_quantity'] = (cv['std'] / cv['mean']).fillna(0)
    return cv[['product_code', 'cv_quantity', 'median_weekly_quantity']]


def _calculate_weekly_txn_median(df_local):
    """Median weekly transaction count based on active days per week."""
    df_local = df_local.copy()
    df_local['is_active_day'] = (df_local['item_quantity'] > 0).astype(int)

    weekly = (
        df_local.set_index('invoice_date')
        .groupby('product_code')['is_active_day']
        .resample('W').sum()
        .reset_index(name='txn_count')
    )
    return (
        weekly.groupby('product_code')['txn_count']
        .median()
        .reset_index(name='median_weekly_transactions')
    )


def _merge_all_features(products_base, core, psd, discounts, cvs, weekly_txns):
    """Joins all feature tables. Drops redundant columns from base to prevent _x/_y suffixes."""
    out = core.copy()
    psd_subset = psd[['product_code', 'product_name', 'duration_days', 'duration_weeks',
                      'share_zero_qty_working_days', 'total_holiday_days', 'total_working_days', 'launch_month']]
    out = out.merge(psd_subset, on='product_code', how='left')

    out = out.merge(weekly_txns, on='product_code', how='left').fillna({'median_weekly_transactions': 0})
    out = out.merge(discounts,   on='product_code', how='left').fillna({'median_discount_rate': 0})
    out = out.merge(cvs,         on='product_code', how='left').fillna({'median_weekly_quantity': 0, 'cv_quantity': 0})

    overlap = [c for c in products_base.columns if c in out.columns and c != 'product_code']

    extra_drops = ['last_sale']
    cols_to_drop = list(set(overlap + extra_drops))

    base_cleaned = products_base.drop(columns=cols_to_drop, errors='ignore')

    return base_cleaned.merge(out, on='product_code', how='left')


def _generate_features_for_split(df_split, products_base):
    """Orchestrates all feature helpers for a single data split."""
    daily = _calculate_daily_product_activity(df_split)
    psd   = _calculate_product_sale_dates(daily, products_base)
    core  = _aggregate_core_features(df_split)
    disc  = _calculate_discount_rate(df_split)
    cvs   = _calculate_cv_quantity(df_split)
    wtxn  = _calculate_weekly_txn_median(df_split)
    return _merge_all_features(products_base, core, psd, disc, cvs, wtxn)

In [ ]:
def _split_timestamps(df_full, BASE_COLS, test_share=0.2, val_share_of_train=0.2):
    """Returns (test_ts, validation_ts) cutoff dates based on product first-sale ordering."""
    products = df_full[BASE_COLS].drop_duplicates().sort_values('first_sale')
    n_total = len(products)

    if n_total == 0:
        return pd.Timestamp.max, pd.Timestamp.max

    # Calculate test cutoff
    if test_share <= 0:
        test_ts = products['first_sale'].max() + pd.Timedelta(days=1)
    elif test_share >= 1:
        test_ts = products['first_sale'].min()
    else:
        test_idx = int(n_total * (1 - test_share))
        test_idx = max(0, min(test_idx, n_total - 1))
        test_ts = products.iloc[test_idx]['first_sale']

    # Calculate validation cutoff from the remaining training/val pool
    train_val = products[products['first_sale'] < test_ts]
    n_train_val = len(train_val)

    if n_train_val > 0:
        if val_share_of_train <= 0:
            validation_ts = train_val['first_sale'].max() + pd.Timedelta(days=1)
        elif val_share_of_train >= 1:
            validation_ts = train_val['first_sale'].min()
        else:
            val_idx = int(n_train_val * (1 - val_share_of_train))
            val_idx = max(0, min(val_idx, n_train_val - 1))
            validation_ts = train_val.iloc[val_idx]['first_sale']
    else:
        validation_ts = test_ts

    return test_ts, validation_ts


def get_df_splits_and_timestamps(df_full, BASE_COLS, test_share=0.2, val_share_of_train=0.2):
    """Splits the full transaction DataFrame into train, validation, and test."""
    test_ts, validation_ts = _split_timestamps(df_full, BASE_COLS, test_share, val_share_of_train)
    df_train = df_full[df_full['invoice_date'] < validation_ts].copy()
    df_val = df_full[(df_full['invoice_date'] >= validation_ts) & (df_full['invoice_date'] < test_ts)].copy()
    df_test = df_full[df_full['invoice_date'] >= test_ts].copy()
    return df_train, df_val, df_test, test_ts, validation_ts


def generate_product_features(df_full, BASE_COLS, test_share=0.2, val_share_of_train=0.2):
    """Generates product-level feature tables for train, validation, and test splits."""
    test_ts, validation_ts = _split_timestamps(df_full, BASE_COLS, test_share, val_share_of_train)

    COLS_TO_DROP = ['product_name', 'price_index', 'last_sale']
    products = df_full[BASE_COLS].drop_duplicates()
    products_clean = products.drop(columns=COLS_TO_DROP, errors='ignore')

    p_train = products_clean[products_clean['first_sale'] < validation_ts].copy()
    p_val = products_clean[(products_clean['first_sale'] >= validation_ts) & (products_clean['first_sale'] < test_ts)].copy()
    p_test = products_clean[products_clean['first_sale'] >= test_ts].copy()

    df_train = df_full[df_full['invoice_date'] < validation_ts]
    df_val = df_full[(df_full['invoice_date'] >= validation_ts) & (df_full['invoice_date'] < test_ts)]
    df_test = df_full[df_full['invoice_date'] >= test_ts]

    return (
        _generate_features_for_split(df_train, p_train) if not p_train.empty else pd.DataFrame(),
        _generate_features_for_split(df_val, p_val) if not p_val.empty else pd.DataFrame(),
        _generate_features_for_split(df_test, p_test) if not p_test.empty else pd.DataFrame(),
    )

In [ ]:
def _prep_df_train_cluster(COLS_TO_DROP_FOR_CLUSTERING, products_with_features_train):
    df_train_cluster = (
        products_with_features_train
        .drop(columns=COLS_TO_DROP_FOR_CLUSTERING, errors='ignore')
        .copy()
    )
    print(f'Clustering feature table: {df_train_cluster.shape[0]:,} products, '
          f'{df_train_cluster.shape[1]} features (excluding product_code)')
    return df_train_cluster


In [ ]:
def preprocess_features(df):
    """
    Scales numeric features and one-hot encodes categoricals.
    product_code is excluded (identifier only).

    Returns
    -------
    product_codes   : DataFrame  — product_code column
    df_preprocessed : DataFrame  — transformed feature matrix
    preprocessor    : fitted ColumnTransformer (call .transform() on val/test)
    """
    df_feats      = df.drop(columns=['product_code'])
    product_codes = df[['product_code']]

    cat_cols = df_feats.select_dtypes(include=['object', 'category']).columns
    num_cols = df_feats.select_dtypes(include=['int64', 'float64']).columns

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
        ],
        remainder='passthrough'
    )

    X = preprocessor.fit_transform(df_feats)

    # Use the preprocessor's built-in method to get exact feature names matching X's shape
    # This resolves the mismatch when 'remainder' columns are included
    feature_names = preprocessor.get_feature_names_out()

    df_preprocessed = pd.DataFrame(X, columns=feature_names, index=df_feats.index)
    return product_codes, df_preprocessed, preprocessor


def plot_elbow_curve(df_preprocessed, k_range=(1, 12)):
    """Plots inertia vs k to aid selection of the optimal number of clusters."""
    K_values = range(k_range[0], k_range[1] + 1)
    inertias = [
        KMeans(n_clusters=k, random_state=42, n_init=10).fit(df_preprocessed).inertia_
        for k in K_values
    ]
    plt.figure(figsize=(10, 5))
    plt.plot(K_values, inertias, marker='o')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Inertia')
    plt.title('Elbow Curve — K-Means Clustering')
    plt.xticks(K_values)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def perform_kmeans_clustering(df_train_cluster, k=3, plot_elbow=False):
    """
    Preprocesses df_train_cluster, fits K-Means, and attaches cluster labels.

    Returns
    -------
    df_clustered : training DataFrame with added 'cluster' column
    preprocessor : fitted ColumnTransformer
    kmeans_model : fitted KMeans
    """
    product_codes, df_preprocessed, preprocessor = preprocess_features(df_train_cluster)
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    df_clustered = df_train_cluster.copy()
    df_clustered['cluster'] = kmeans.fit_predict(df_preprocessed)

    if plot_elbow:
        plot_elbow_curve(df_preprocessed)

    return df_clustered, preprocessor, kmeans


In [ ]:
flag_run = False

if flag_run:
    # Swap in any k's result if needed:
    # df_clustered = all_clustered_results[5]

    num_cols_cl = df_clustered.select_dtypes(include=['float64', 'int64']).columns.drop('cluster', errors='ignore')

    print(f"Cluster sizes (k={df_clustered['cluster'].nunique()}):")
    display(df_clustered['cluster'].value_counts().sort_index())

    print('\n--- Cluster Mean of Numerical Features ---')
    display(df_clustered.groupby('cluster')[num_cols_cl].mean().round(3))

    print('\n--- Region Distribution by Cluster (%) ---')
    display(pd.crosstab(df_clustered['cluster'], df_clustered['Region'], normalize='index').mul(100).round(1))

    top5 = df_clustered['full_sub_cat'].value_counts().nlargest(5).index
    df_plot = df_clustered.copy()
    df_plot['full_sub_cat_grouped'] = df_plot['full_sub_cat'].where(df_plot['full_sub_cat'].isin(top5), 'Other')
    print('\n--- Top-5 Sub-Category Distribution by Cluster (%) ---')
    display(pd.crosstab(df_plot['cluster'], df_plot['full_sub_cat_grouped'], normalize='index').mul(100).round(1))

    features_to_plot = ['median_weekly_quantity', 'median_weekly_transactions',
                        'cv_quantity', 'price_index', 'share_zero_qty_working_days']
    fig, axes = plt.subplots(3, 2, figsize=(12, 12))
    for ax, feat in zip(axes.flat, features_to_plot):
        sns.boxplot(x='cluster', y=feat, data=df_clustered, ax=ax)
        ax.set_title(feat)
    fig.delaxes(axes.flat[-1])
    plt.tight_layout()
    plt.show()

    pd.crosstab(df_clustered['cluster'], df_clustered['Region'], normalize='index').plot(
        kind='bar', stacked=True, figsize=(10, 5)
    )
    plt.title('Region Distribution by Cluster')
    plt.ylabel('Proportion')
    plt.legend(title='Region', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()


In [ ]:
def wmape(actual, predicted):
    """Weighted Mean Absolute Percentage Error. Returns NaN if sum(actual) == 0."""
    actual    = np.array(actual,    dtype=float)
    predicted = np.array(predicted, dtype=float)
    total = actual.sum()
    return np.nan if total == 0 else np.abs(actual - predicted).sum() / total


def find_best_label_mapping(true_labels, pred_labels):
    """
    Finds the permutation of pred_labels that maximises agreement with true_labels.
    Uses the Hungarian algorithm on the confusion matrix.
    Requires both arrays to have the same number of unique values.
    """
    unique_true = np.unique(true_labels)
    unique_pred = np.unique(pred_labels)

    t_map = {v: i for i, v in enumerate(unique_true)}
    p_map = {v: i for i, v in enumerate(unique_pred)}
    mapped_true = np.array([t_map[l] for l in true_labels])
    mapped_pred = np.array([p_map[p] for p in pred_labels])  # note: iterate p, index p_map[p]

    cm = confusion_matrix(mapped_true, mapped_pred)
    row_ind, col_ind = linear_sum_assignment(-cm)  # negative to maximise

    best_map = {unique_pred[col_ind[i]]: unique_true[row_ind[i]] for i in range(len(row_ind))}
    return np.array([best_map.get(p, -1) for p in pred_labels])  # -1 = unmapped


def calculate_wmape_baseline(df_train_full, df_validation_full, df_clustered_k, new_products_val_k):
    """
    Cluster-Mean Baseline: compute 8-week total forecast per cluster from training data,
    evaluate against actual 8-week totals of new validation products.

    Parameters
    ----------
    df_train_full      : full training transaction DataFrame
    df_validation_full : full validation transaction DataFrame
    df_clustered_k     : training product DataFrame with 'cluster' column
    new_products_val_k : validation new-product DataFrame with 'predicted_cluster_classifier'

    Returns
    -------
    overall_wmape        : float
    detailed_val_results : DataFrame — one row per product with cluster, baseline, actual
    """
    # ── Step A: 8-week total per training product ───────────────────────────
    train_weekly = (
        df_train_full.set_index('invoice_date')
        .groupby('product_code')['item_quantity']
        .resample('W').sum()
        .reset_index()
    )
    train_weekly['week_num'] = train_weekly.groupby('product_code').cumcount() + 1
    train_8w = (
        train_weekly[train_weekly['week_num'] <= 8]
        .groupby('product_code')['item_quantity']
        .sum()
        .reset_index(name='total_8w_qty')
    )
    train_8w = train_8w.merge(
        df_clustered_k[['product_code', 'cluster']], on='product_code', how='inner'
    )

    # ── Step B: Cluster baseline = mean of 8-week totals, rounded to integer ──
    cluster_baseline = (
        train_8w.groupby('cluster')['total_8w_qty']
        .mean()
        .round(0)                         # round to nearest whole unit
        .astype(int)                      # store as integer
        .rename('baseline_total_8w_qty')
        .reset_index()
    )

    # ── Step C: Actual 8-week totals for new validation products ───────────
    new_val_codes = new_products_val_k['product_code'].unique()
    val_weekly = (
        df_validation_full[df_validation_full['product_code'].isin(new_val_codes)]
        .set_index('invoice_date')
        .groupby('product_code')['item_quantity']
        .resample('W').sum()
        .reset_index()
    )
    val_weekly['week_num'] = val_weekly.groupby('product_code').cumcount() + 1
    val_8w = (
        val_weekly[val_weekly['week_num'] <= 8]
        .groupby('product_code')['item_quantity']
        .sum()
        .reset_index(name='actual_total_8w_qty')
    )

    # ── Step D: Attach cluster assignment and baseline ──────────────────────
    val_8w = val_8w.merge(
        new_products_val_k[['product_code', 'predicted_cluster_classifier']].rename(
            columns={'predicted_cluster_classifier': 'cluster'}
        ),
        on='product_code', how='left'
    )
    val_8w = val_8w.merge(cluster_baseline, on='cluster', how='left')

    # If a cluster has no training history, default baseline to 0
    val_8w['baseline_total_8w_qty'] = val_8w['baseline_total_8w_qty'].fillna(0).astype(int)

    overall_wmape = wmape(val_8w['actual_total_8w_qty'], val_8w['baseline_total_8w_qty'])
    return overall_wmape, val_8w


In [ ]:
def _prep_feed_loop(products_with_features_val, products_with_features_train, df_train_cluster, EXCLUDE_FROM_CLASSIFIER):
    # New products: appear in validation but not in training
    new_products_val_df = products_with_features_val[
        ~products_with_features_val['product_code'].isin(products_with_features_train['product_code'])
    ].copy()
    print(f'New products in validation set: {len(new_products_val_df):,}')

    clustering_features = [col for col in df_train_cluster.columns if col != 'product_code']

    CLASSIFIER_FEATURES = [
        f for f in clustering_features if f not in EXCLUDE_FROM_CLASSIFIER
    ]
    return new_products_val_df, CLASSIFIER_FEATURES

# # Ensure is_new_brand is in the classifier features if it exists in the training set
# if 'is_new_brand' in df_train_cluster.columns and 'is_new_brand' not in CLASSIFIER_FEATURES:
#     CLASSIFIER_FEATURES.append('is_new_brand')

def run_single_k(k_val, df_train_cluster_in, new_products_val_df, df_train_full_in, df_validation_full_in, CLASSIFIER_FEATURES):
    """Worker function to process a single k in parallel."""

    # 1. Fit K-Means
    df_clustered, _, kmeans_model = perform_kmeans_clustering(df_train_cluster_in, k=k_val, plot_elbow=False)

    # 2. Train Logistic Regression
    X_train_cls = df_clustered[CLASSIFIER_FEATURES].copy()
    y_train_cls = df_clustered['cluster']

    num_cls_features = X_train_cls.select_dtypes(include=np.number).columns.tolist()
    cat_cls_features = X_train_cls.select_dtypes(exclude=np.number).columns.tolist()

    cls_pipeline = Pipeline([
        ('preprocessor', ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), num_cls_features),
                ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cls_features)
            ],
            remainder='passthrough'
        )),
        ('classifier', LogisticRegression(max_iter=1000, random_state=42))
    ])
    cls_pipeline.fit(X_train_cls, y_train_cls)

    # 3. Classify
    X_val_cls = new_products_val_df[CLASSIFIER_FEATURES].copy()
    classifier_preds_val = cls_pipeline.predict(X_val_cls)
    new_products_val_k = new_products_val_df.copy()
    new_products_val_k['predicted_cluster_classifier'] = classifier_preds_val

    # 4. Evaluate
    overall_wmape, detailed_val_results = calculate_wmape_baseline(
        df_train_full_in, df_validation_full_in, df_clustered, new_products_val_k
    )

    act = detailed_val_results['actual_total_8w_qty']
    pred = detailed_val_results['baseline_total_8w_qty']

    rmse_val = np.sqrt(mean_squared_error(act, pred))
    mae_val = mean_absolute_error(act, pred)
    bias_val = (pred.sum() - act.sum()) / act.sum() if act.sum() != 0 else np.nan

    return {
        'k': k_val,
        'wmape': overall_wmape,
        'rmse': rmse_val,
        'mae': mae_val,
        'bias': bias_val,
        'detailed_results': detailed_val_results
    }


def run_loop(
    k_min, k_max, df_train_cluster,
    new_products_val_df, df_train_full_in,
    df_validation_full_in,
    CLASSIFIER_FEATURES
    ):

    # Run the sweep
    num_cores = multiprocessing.cpu_count()
    parallel_results = Parallel(n_jobs=num_cores, verbose=10)(
        delayed(run_single_k)(k, df_train_cluster, new_products_val_df, df_train_full_in, df_validation_full_in, CLASSIFIER_FEATURES)
        for k in range(k_min, k_max)
    )
    all_detailed_val_results = {res['k']: res['detailed_results'] for res in parallel_results}
    results = [{k: v for k, v in res.items() if k != 'detailed_results'} for res in parallel_results]
    results_df = pd.DataFrame(results)
    display(results_df)
    sorted_results = results_df.sort_values(by=['wmape', 'k'])
    sort_round_results = sorted_results.copy().round(2).sort_values(by=['wmape', 'k'])
    opt_k = int(sort_round_results.iloc[0]['k'])
    opt_k_detailed_val_results = all_detailed_val_results[opt_k]

    print(f"\nSelection complete. Optimal k: {opt_k}")
    display(sorted_results[['k', 'wmape', 'rmse', 'mae', 'bias']].head(10))

    return all_detailed_val_results, sorted_results, opt_k, opt_k_detailed_val_results

In [ ]:
def _features_show(df_train_cluster, CLASSIFIER_FEATURES):
    clustering_features = [col for col in df_train_cluster.columns if col != 'product_code']
    classifier_features = CLASSIFIER_FEATURES

    print(f"--- Features used for Clustering (k-means) [{len(clustering_features)}] ---")
    print(clustering_features)

    print(f"\n--- Features used for Classification (Logistic Regression) [{len(classifier_features)}] ---")
    print(classifier_features)

    # Identifying the overlap/differences
    only_clustering = set(clustering_features) - set(classifier_features)
    only_classifier = set(classifier_features) - set(clustering_features)

    if only_clustering:
        print(f"\nFeatures in Clustering but NOT in Classifier:\n{list(only_clustering)}")
    if only_classifier:
        print(f"\nFeatures in Classifier but NOT in Clustering:\n{list(only_classifier)}")

In [ ]:
def wrapper(
    df, k_min, k_max,
    FORECAST_HORIZON_WEEKS,
    COLS2DROP_BASE,
    BASE_COLS,
    COLS_TO_DROP_FOR_CLUSTERING,
    EXCLUDE_FROM_CLASSIFIER,
    test_share=0.2,
    val_share_of_train=0.2,
    flag_export = False
    ):

  # INITIAL PREP
  df = _prep_df(df, COLS2DROP_BASE, FORECAST_HORIZON_WEEKS)


  # Run splits
  df_train_full, df_validation_full, df_test_full, test_ts, validation_ts = (
      get_df_splits_and_timestamps(df, BASE_COLS, test_share=test_share, val_share_of_train=val_share_of_train)
  )
  products_with_features_train, products_with_features_val, products_with_features_test = (
      generate_product_features(df, BASE_COLS, test_share=test_share, val_share_of_train=val_share_of_train)
  )

  print(f'Train products      : {products_with_features_train.shape[0]:,}')
  print(f'Validation products : {products_with_features_val.shape[0]:,}')
  print(f'Test products       : {products_with_features_test.shape[0]:,}')
  print(f'Validation cutoff   : {validation_ts.date()}')
  print(f'Test cutoff         : {test_ts.date()}')

  # TRAIN DF for CLUSTER
  df_train_cluster = _prep_df_train_cluster(COLS_TO_DROP_FOR_CLUSTERING, products_with_features_train)

  # CORR CHECK
  num_cols = df_train_cluster.select_dtypes(include=np.number).columns.tolist()

  corr = df_train_cluster[num_cols].corr(method='spearman')  # Spearman is more robust
  sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')

  # `median_weekly_quantity` and `median_weekly_transactions` are 100% correlated! -> drop `median_weekly_transactions`
  # `cv_quantity` and `share_zero_qty_working_days` are 100% correlated! -> drop `share_zero_qty_working_days`

  df_train_cluster.drop(columns=['median_weekly_transactions', 'share_zero_qty_working_days'], errors='ignore', inplace=True)

  # ELBOW PLOT
  _, df_preprocessed_for_elbow, _ = preprocess_features(df_train_cluster)
  plot_elbow_curve(df_preprocessed_for_elbow, k_range=(1, 12))

  new_products_val_df, CLASSIFIER_FEATURES = _prep_feed_loop(
      products_with_features_val, products_with_features_train, df_train_cluster, EXCLUDE_FROM_CLASSIFIER
  )

  all_detailed_val_results, sorted_results, opt_k, opt_k_detailed_val_results = run_loop(
      k_min, k_max, df_train_cluster,
      new_products_val_df, df_train_full,
      df_validation_full,
      CLASSIFIER_FEATURES
  )

  _features_show(df_train_cluster, CLASSIFIER_FEATURES)

  if flag_export:
    # Two files are exported:
    # 1. **Macro results** — one row per k, all metrics
    # 2. **Detailed results for optimal k** — one row per new validation product.

    france_timezone = pytz.timezone('Europe/Paris')
    current_time    = datetime.now(france_timezone).strftime('%Y%m%d_%H%M')

    # ── Macro results (one row per k) ───────────────────────────────────────────
    macro_path = out_folder / f'clustering_forecasting_results_{current_time}.xlsx'
    sorted_results.to_excel(macro_path, index=False)
    print(f'Macro results exported to: {macro_path}')
    display(sorted_results)

    # ── Detailed results for optimal k ──────────────────────────────────────────
    # Columns: product_code | cluster | baseline_total_8w_qty | actual_total_8w_qty
    detailed_path = out_folder / f'clustering_forecasting_results_detailed_k{opt_k}_{current_time}.xlsx'
    opt_k_detailed_val_exp = opt_k_detailed_val_results[['product_code', 'cluster', 'baseline_total_8w_qty', 'actual_total_8w_qty']].copy()
    opt_k_detailed_val_exp.to_excel(detailed_path, index=False)
    print(f'Detailed results (k={opt_k}) exported to: {detailed_path}')

  return all_detailed_val_results, sorted_results, opt_k, opt_k_detailed_val_results

In [ ]:
k_min = 2
k_max = 100

test_share=0.2
val_share_of_train=0.2

FORECAST_HORIZON_WEEKS = 8

COLS2DROP_BASE = [
    'is_corrected_cq', 'is_corrected_cq_opposite', 'price_level',
    'is_corrected_only_price', 'last_sale', 'sub cat', 'sub sub cat',
]

BASE_COLS = [
    'product_code', 'product_name', 'brand_name', 'Producer', 'Region',
    'price_index', 'Category', 'full_sub_sub_cat',
    'num_similars', 'substitutability', 'first_sale', 'full_sub_cat', 'is_new_brand'
]

COLS_TO_DROP_FOR_CLUSTERING = [
    'product_name',
    'sub cat', 'sub sub cat',
    'first_sale',
    # 'total_sales_quantity',
    # 'duration_weeks',
    # 'total_working_days',
    # 'num_sales_transactions',
    ]

EXCLUDE_FROM_CLASSIFIER = [
    'num_sales_transactions', 'duration_days', 'duration_weeks',
    'share_zero_qty_working_days', 'total_holiday_days', 'total_working_days', 'total_sales_quantity',
    'median_weekly_transactions', 'median_discount_rate', 'cv_quantity', 'median_weekly_quantity'
]

# LOAD DATA
df = pd.read_pickle(folder_path / 'final_merged_dataset.pkl')
print(f'Loaded {len(df):,} rows, {df.shape[1]} columns.')

## VAL SET

In [ ]:
all_detailed_val_results, sorted_results, opt_k, opt_k_detailed_val_results = wrapper(
    df, k_min, k_max,
    FORECAST_HORIZON_WEEKS,
    COLS2DROP_BASE,
    BASE_COLS,
    COLS_TO_DROP_FOR_CLUSTERING,
    EXCLUDE_FROM_CLASSIFIER,
    test_share=test_share,
    val_share_of_train=val_share_of_train,
    flag_export=True
    )

## TEST SET

In [ ]:
all_detailed_val_results, sorted_results, opt_k, opt_k_detailed_val_results = wrapper(
    df, opt_k, (opt_k+1),
    FORECAST_HORIZON_WEEKS,
    COLS2DROP_BASE,
    BASE_COLS,
    COLS_TO_DROP_FOR_CLUSTERING,
    EXCLUDE_FROM_CLASSIFIER,
    test_share=0,
    val_share_of_train=0.2,
    flag_export=True
    )